# Практические работы 1-7 по R
**Студент:** Токарев  
**Репозиторий:** [OXOTAHAAPTEMACOBA2210/Pz_Tokarev](https://github.com/OXOTAHAAPTEMACOBA2210/Pz_Tokarev)

Этот блокнот полностью настроен для запуска в **Google Colab** (ядро R).
Нажимайте кнопку Play последовательно по ячейкам.


## 0. Скачивание файлов данных и установка пакетов


In [ ]:
# 1. Скачиваем репозиторий с датасетами (bmw.csv, Employee.csv) и скриптами
if (!file.exists("bmw.csv")) {
  system("git clone https://github.com/OXOTAHAAPTEMACOBA2210/Pz_Tokarev.git temp_repo")
  file.copy(list.files("temp_repo", full.names = TRUE), ".", recursive = TRUE)
  unlink("temp_repo", recursive = TRUE)
}
list.files()


In [ ]:
# 2. Установка необходимых библиотек для графики, статистики и ML
pkgs <- c("ggplot2", "psych", "ROCR", "caTools", "party", "randomForest", "e1071")
new_pkgs <- pkgs[!(pkgs %in% installed.packages()[, "Package"])]
if (length(new_pkgs)) install.packages(new_pkgs, repos = "https://cloud.r-project.org")
cat("Библиотеки готовы к работе!\n")


## ПЗ 1. Основы работы с R


In [ ]:
# Занятие 1. Основы работы с R

# 1. Поголовье животных
cattle <- 3456
pig_population <- 7333
chicken_population <- 23459

meat_cows <- 1320
milk_cows <- cattle - meat_cows
milk_cows

egg_chickens <- 12457
meat_chickens <- chicken_population - egg_chickens
meat_chickens

# 2. Перевозка свиней
total_pigs <- 1457
capacity <- 14

trucks <- total_pigs %/% capacity
trucks

remaining_pigs <- total_pigs %% capacity
remaining_pigs

# 3. Тестовые коровы и надой
milk_per_day <- c(12, 4, 7, 18, 10)
annual_milk <- milk_per_day * 365
annual_milk

mean_annual <- mean(annual_milk)
mean_annual
# средний надой 3723 литра, это меньше 5000-7000 л, коровы не высокопродуктивные

# 4. Датафрейм с коровами
cows_names <- c("Ералаш", "Пальто", "Кощей", "Адидас", "Зима")
cows_df <- data.frame(
  Name = cows_names,
  Daily_Yield = milk_per_day,
  Annual_Yield = annual_milk
)
cows_df


## ПЗ 2. Работа с датафреймами (Employee.csv)


In [ ]:
# Занятие 2. Работа с Дата фреймами (Employee.csv)

# 1. Загрузка данных и структура
df <- read.csv("Employee.csv", stringsAsFactors = FALSE)
nrow(df)
ncol(df)
str(df)

# уникальные значения качественных признаков
unique(df$Education)
unique(df$City)
unique(df$Gender)
unique(df$EverBenched)

# 2. summary для данных
summary(df)

# 3. Сотрудники старше и младше среднего возраста
mean_age <- mean(df$Age)
mean_age
older <- df[df$Age > mean_age, ]
younger <- df[df$Age < mean_age, ]
head(older)
head(younger)

# 4. Присоединились до 2016 года (пол, возраст, стаж)
early_emp <- df[df$JoiningYear < 2016, c("Gender", "Age", "ExperienceInCurrentDomain")]
head(early_emp)

# 5. Средний уровень оплаты и возраст уволившихся
mean(df$PaymentTier[df$LeaveOrNot == 1])
mean(df$Age[df$LeaveOrNot == 1])

# 6. Мин и макс стаж для PaymentTier == 3
min(df$ExperienceInCurrentDomain[df$PaymentTier == 3])
max(df$ExperienceInCurrentDomain[df$PaymentTier == 3])
range(df$ExperienceInCurrentDomain[df$PaymentTier == 3])

# 7. Увольнения среди мужчин и женщин
table(df$Gender, df$LeaveOrNot)
prop.table(table(df$Gender, df$LeaveOrNot), 1)

# 8. Различия в уровне оплаты
table(df$Gender, df$PaymentTier)

# 9. Город регистрации большинства
table(df$City)

# 10. Средний возраст тех, кто раньше всех пришел в компанию
min_year <- min(df$JoiningYear)
mean(df$Age[df$JoiningYear == min_year])

# 11. Уровень образования
table(df$Education)

# 12. Образование и пол
table(df$Education, df$Gender)

# 13. Кто чаще увольнялся
table(df$Education, df$LeaveOrNot)
prop.table(table(df$Education, df$LeaveOrNot), 1)

# 14. Разделение датасета по LeaveOrNot
df_stayed <- subset(df, LeaveOrNot == 0)
df_left <- subset(df, LeaveOrNot == 1)
nrow(df_stayed)
nrow(df_left)


## ПЗ 3. Описательные статистики и графики (bmw.csv)


In [ ]:
# Занятие 3. Описательные статистики и графики

library(psych)
library(ggplot2)

# Загружаем датасет с маркой BMW
df <- read.csv("bmw.csv", stringsAsFactors = TRUE)
str(df)

# 1. Описательные статистики для цены
mean(df$price)
median(df$price)
sd(df$price)
range(df$price)

# 2. Средняя стоимость по типу топлива
aggregate(price ~ fuelType, data = df, FUN = mean)

# 3. Средняя стоимость по типу топлива и коробке передач
aggregate(price ~ fuelType + transmission, data = df, FUN = mean)

# 4. Описательная статистика через psych::describe
describe(df)
describe(df[, c("price", "mpg")])

# 5. Стандартные графики R
# гистограмма цены
hist(df$price, breaks = 20, col = "lightblue", main = "Гистограмма стоимости BMW", xlab = "Цена ($)", ylab = "Частота")

# ящик с усами
boxplot(price ~ transmission, data = df, main = "Стоимость по типу коробки передач", xlab = "Коробка передач", ylab = "Цена ($)")

# точечная диаграмма
plot(df$mileage, df$price, pch = 16, col = rgb(0, 0, 1, 0.2), main = "Цена от пробега", xlab = "Пробег", ylab = "Цена")

# 6. Гистограмма через ggplot2 для расхода (mpg)
ggplot(df, aes(x = mpg)) +
  geom_histogram(binwidth = 3, fill = "steelblue", color = "white") +
  theme_minimal() +
  labs(title = "Гистограмма расхода топлива", x = "Расход (миль на галлон)", y = "Количество")

# 7. График плотности стоимости по типу топлива
ggplot(subset(df, fuelType %in% c("Diesel", "Petrol", "Hybrid")), aes(x = price, fill = fuelType)) +
  geom_density(alpha = 0.5) +
  theme_minimal() +
  labs(title = "Плотность распределения стоимости по топливу", x = "Цена", y = "Плотность")

# 8. Точечная диаграмма: цена от пробега, цвет - топливо, размер - объем двигателя
ggplot(subset(df, fuelType %in% c("Diesel", "Petrol", "Hybrid")), 
       aes(x = mileage, y = price, color = fuelType, size = engineSize)) +
  geom_point(alpha = 0.4) +
  theme_minimal() +
  labs(title = "Зависимость стоимости от пробега и объема двигателя", x = "Пробег", y = "Цена")


## ПЗ 4. Корреляция и линейная регрессия (bmw.csv)


In [ ]:
# Занятие 4. Корреляция и линейная регрессия

library(psych)
library(ggplot2)

df <- read.csv("bmw.csv", stringsAsFactors = TRUE)

# 1. Корреляция цены и пробега
cor.test(df$mileage, df$price)

# 2-3. Датасет только с численными переменными и парные диаграммы
df_numeric <- df[, c("year", "price", "mileage", "tax", "mpg", "engineSize")]
pairs(df_numeric)

# 4. Матрица корреляций и проверка значимости
cor(df_numeric)
fit_corr <- corr.test(df_numeric)
fit_corr$r
fit_corr$p

# 5. Парная линейная регрессия цены от пробега
fit <- lm(price ~ mileage, data = df)
fit
summary(fit)

# 6. Точечная диаграмма с линией тренда
ggplot(df, aes(x = mileage, y = price)) +
  geom_point(alpha = 0.25) +
  geom_smooth(method = "lm", col = "red") +
  theme_minimal() +
  labs(title = "Линейная регрессия стоимости от пробега", x = "Пробег", y = "Цена")

# 7. Добавление факторов (коробка передач и тип топлива)
ggplot(df, aes(x = mileage, y = price, col = transmission)) +
  geom_point(alpha = 0.25) +
  geom_smooth(method = "lm") +
  theme_minimal() +
  labs(title = "Регрессия по типу КПП", x = "Пробег", y = "Цена")

ggplot(subset(df, fuelType %in% c("Diesel", "Petrol", "Hybrid")), aes(x = mileage, y = price, col = fuelType)) +
  geom_point(alpha = 0.25) +
  geom_smooth(method = "lm") +
  theme_minimal() +
  labs(title = "Регрессия по типу топлива", x = "Пробег", y = "Цена")

# 8. Прогноз по модели
new_mileage <- data.frame(mileage = c(1564, 32587, 9834, 343445, 10045))
predict(fit, new_mileage)


## ПЗ 5. Множественная линейная регрессия (bmw.csv)


In [ ]:
# Занятие 5. Множественная линейная регрессия

df <- read.csv("bmw.csv", stringsAsFactors = TRUE)

# 1. Множественная регрессия от года и пробега
fit1 <- lm(price ~ year + mileage, data = df)
fit1
summary(fit1)

# 2. Модель со всеми факторами
fit_all <- lm(price ~ ., data = df)
summary(fit_all)

# Исключаем налог и расход для проверки через ANOVA
fit_reduced <- lm(price ~ model + year + transmission + mileage + fuelType + engineSize, data = df)
anova(fit_all, fit_reduced)

# 3. Пошаговое исключение факторов
opt_model <- step(fit_all, direction = "backward")
summary(opt_model)


## ПЗ 6. Логистическая регрессия (Employee.csv)


In [ ]:
# Занятие 6. Логистическая регрессия

library(ggplot2)
library(ROCR)

# 1. Загрузка данных
df <- read.csv("Employee.csv", stringsAsFactors = TRUE)
str(df)

# 2. Точечная диаграмма
ggplot(df, aes(x = Age, y = PaymentTier, col = factor(LeaveOrNot), shape = Gender)) +
  geom_jitter(width = 0.3, height = 0.15, alpha = 0.5) +
  theme_minimal() +
  labs(title = "Увольнение в зависимости от возраста, зарплаты и пола", 
       x = "Возраст", y = "Грейд зарплаты", col = "LeaveOrNot")

# 3. Модель логистической регрессии
fit <- glm(LeaveOrNot ~ Gender + Age + PaymentTier, data = df, family = "binomial")
summary(fit)
exp(fit$coefficients)

# 4. Предсказанные вероятности
df$prob <- predict(fit, type = "response")
head(df[, c("Gender", "Age", "PaymentTier", "LeaveOrNot", "prob")])

# 5. ROC-кривая
pred_fit <- prediction(df$prob, df$LeaveOrNot)
perf_fit <- performance(pred_fit, "tpr", "fpr")
plot(perf_fit, colorize = TRUE, print.cutoffs.at = seq(0, 1, by = 0.1))

# 6. Специфичность и чувствительность
perf_spec <- performance(pred_fit, x.measure = "cutoff", measure = "spec")
perf_sens <- performance(pred_fit, x.measure = "cutoff", measure = "sens")
perf_acc  <- performance(pred_fit, x.measure = "cutoff", measure = "acc")

plot(perf_spec, col = "red", lwd = 2)
plot(perf_sens, col = "green", lwd = 2, add = TRUE)
plot(perf_acc,  col = "black", lwd = 2, add = TRUE)

# точка пересечения чувствительности и специфичности
cutoffs <- perf_spec@x.values[[1]]
spec_vals <- perf_spec@y.values[[1]]
sens_vals <- perf_sens@y.values[[1]]
cutoff <- cutoffs[which.min(abs(spec_vals - sens_vals))]
cutoff

# 7. Классификация по выбранному порогу
df$pred_resp <- ifelse(df$prob >= cutoff, 1, 0)

# 8. Сравнение с фактом и расчет точности
df$correct <- ifelse(df$pred_resp == df$LeaveOrNot, 1, 0)
table(Факт = df$LeaveOrNot, Прогноз = df$pred_resp)
accuracy <- sum(df$correct) / nrow(df)
accuracy


## ПЗ 7. Методы машинного обучения (iris: Дерево решений, Random Forest, SVM)


In [ ]:
# Занятие 7. Методы машинного обучения

library(caTools)
library(party)
library(randomForest)
library(e1071)

# 1. Загрузка данных iris и разбивка 70/30
df <- iris

set.seed(42)
sample <- sample.split(df$Species, SplitRatio = 0.7)
train <- subset(df, sample == TRUE)
test  <- subset(df, sample == FALSE)

# 2.1 Дерево решений
model <- ctree(Species ~ ., train)
plot(model)

predict_model <- predict(model, test)
m_at <- table(test$Species, predict_model)
m_at
ac_Test <- sum(diag(m_at)) / sum(m_at)
ac_Test

# 2.2 Случайный лес
classifier_RF <- randomForest(x = train[-5], y = train$Species, ntree = 500)
classifier_RF
plot(classifier_RF)

importance(classifier_RF)

y_pred <- predict(classifier_RF, newdata = test[-5])
confusion_mtx <- table(test[, 5], y_pred)
confusion_mtx
ac_Test2 <- sum(diag(confusion_mtx)) / sum(confusion_mtx)
ac_Test2

# 2.3 Метод опорных векторов (SVM)
classifier <- svm(formula = Species ~ ., data = train, type = "C-classification", kernel = "linear")
classifier

y_pred3 <- predict(classifier, newdata = test[-5])
cm <- table(test[, 5], y_pred3)
cm
ac_Test3 <- sum(diag(cm)) / sum(cm)
ac_Test3

# 3. Итоговая таблица точности
results <- data.frame(
  Model = c("Дерево решений", "Случайный лес", "SVM"),
  Accuracy = c(ac_Test, ac_Test2, ac_Test3)
)
results

# 4. Наиболее точный метод
results[which.max(results$Accuracy), ]
